# 2-3 集計と絞り込み

前回読み込んだ売上データから、**「商品ごとの売上はいくら？」「1万円以上の取引だけ取り出したい」** といった、実務でよくある集計・絞り込みを pandas で一気に書きます。

Excel で言えば **ピボットテーブル と オートフィルタ** に当たる部分です。

## このノートのゴール

- `groupby()` で「○○ごとの合計／平均／件数」を 1 行で出せる
- `.agg()` で複数の集計を同時に出せる
- ブール条件 (`df[df["列"] >= 1000]`) で行を絞り込める
- 複数条件（`&` `|`）、`isin()`、`between()` を使い分けられる
- 1-4 で書いた `for` ループ集計を、pandas で 1〜2 行に書き直せる

## Excel / VBA との対応表

| やりたいこと | Excel / VBA | pandas |
|---|---|---|
| 商品ごとの売上合計 | ピボットテーブル（行: 商品、値: 売上合計） | `df.groupby("product_name")["amount"].sum()` |
| 1万円以上だけ表示 | オートフィルタ → 数値フィルタ | `df[df["amount"] >= 10000]` |
| 特定の商品コードだけ | フィルタ → リストから選択 | `df[df["product_code"].isin(["P001", "P004"])]` |
| 1万〜2万円の範囲 | フィルタ → 指定の値の間 | `df[df["amount"].between(10000, 20000)]` |
| 商品×日付の集計 | ピボット（行: 商品、列: 日付） | `df.groupby(["product_name", "date"])["amount"].sum()` |

**ポイント**: pandas では、ピボットテーブルもオートフィルタも **1 行のコード** で書けます。一度書けば、データが入れ替わっても同じコードで動きます。

## データの準備

2-2 と同じ売上データ (`sales_2026-01.xlsx`) を読み込みます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
BASE = "/content/drive/MyDrive/python-data-basics"
DATA = f"{BASE}/data/pandas"

df = pd.read_excel(f"{DATA}/sales_2026-01.xlsx", sheet_name="売上", parse_dates=["date"])
print(f"{len(df)} 件のデータを読み込みました")
df.head()

## 1. `groupby()` で「○○ごとの集計」

**ピボットテーブルの「行に商品、値に売上合計」と同じことを 1 行で書きます。**

書き方:
```python
df.groupby("グループ化したい列")["集計したい列"].sum()
```

In [ ]:
# 商品ごとの売上合計
df.groupby("product_name")["amount"].sum()

## 2. いろいろな集計関数

`.sum()` の部分を入れ替えるだけで、合計以外の集計もできます。

| 関数 | 意味 | Excel での対応 |
|---|---|---|
| `.sum()` | 合計 | `SUMIF` |
| `.mean()` | 平均 | `AVERAGEIF` |
| `.count()` | 件数 | `COUNTIF` |
| `.max()` | 最大 | `MAXIFS` |
| `.min()` | 最小 | `MINIFS` |

In [ ]:
# 商品ごとの「取引件数」
print("=== 商品ごとの取引件数 ===")
print(df.groupby("product_name")["amount"].count())

print("\n=== 商品ごとの平均売上 ===")
print(df.groupby("product_name")["amount"].mean().round(0))

print("\n=== 商品ごとの最大取引額 ===")
print(df.groupby("product_name")["amount"].max())

## 3. `.agg()` で複数の集計を同時に

「件数と合計と平均を一覧で見たい」というときは `.agg()` にリストで渡します。

これも Excel のピボットだと、値フィールドを 3 つ追加することに相当します。

In [ ]:
# 商品ごとの「件数・合計・平均」を一覧で
df.groupby("product_name")["amount"].agg(["count", "sum", "mean"]).round(0)

## 4. ブール条件で行を絞り込む

**Excel のオートフィルタに相当します。**

書き方:
```python
df[ 条件 ]
```

`条件` には `df["列名"] >= 1000` のような **「列全体に対する比較式」** を書きます。

比較式の結果は `True` / `False` の列になり、`True` の行だけが残ります。

In [ ]:
# 5000円以上の取引だけ取り出す
df[df["amount"] >= 5000]

### 中身を分解してみる

`df["amount"] >= 5000` は、True/False の列を返します。それを `df[ ... ]` に渡すと、True の行だけが残るしくみです。

In [ ]:
# 条件式そのものを見てみる
mask = df["amount"] >= 5000
print(mask.head(10))  # True/False の列
print(f"\nTrue の数: {mask.sum()} 件 / 全 {len(mask)} 件")

## 5. 複数条件・特殊な絞り込み

| やりたいこと | 書き方 |
|---|---|
| かつ (AND) | `(条件A) & (条件B)` |
| または (OR) | `(条件A) \| (条件B)` |
| 特定のリスト内 | `df["列"].isin([値1, 値2, ...])` |
| 範囲内 | `df["列"].between(下限, 上限)` |

**注意**: 複数条件のときは、各条件を **必ず `()` で囲む** こと。Python 的には `and` / `or` ではなく **`&` / `|`** を使います（よく忘れます）。

In [ ]:
# (A) りんご かつ 1000円以上
print("=== りんご かつ 1000円以上 ===")
print(df[(df["product_name"] == "りんご (1袋)") & (df["amount"] >= 1000)])

# (B) 商品コードが P001 または P004
print("\n=== P001 または P004 ===")
print(df[df["product_code"].isin(["P001", "P004"])].head())

# (C) 売上が 1000〜5000円の範囲
print("\n=== 売上が 1000〜5000円 ===")
print(df[df["amount"].between(1000, 5000)].head())

## 6. 1-4 の `for` ループを pandas で書き直す

1-4 では、「1万円以上の件数」を `for` ループで数えました:

```python
# 1-4 で書いたコード
count = 0
for s in sales:
    if s >= 10000:
        count += 1
print(count)
```

**pandas なら 1 行です。**

In [ ]:
# 1-4 で書いたループ集計が、pandas なら 1 行
(df["amount"] >= 10000).sum()

## 練習問題

それぞれ **コード 1〜2 行** で書けます。下のセルに書いて実行してみてください。

1. **商品ごとの「最大の取引金額」** を出してください
2. **「ぶどう (1パック)」の取引だけ** を抽出して、合計金額を出してください
3. **1月15日以降の取引** だけを抽出してください（ヒント: `df["date"] >= "2026-01-15"`）
4. **「メロン」または「いちご (1パック)」** で、かつ **数量が 5 個以上** の取引を抽出してください

In [ ]:
# ここに書いてみてください



<details>
<summary>答えを見る</summary>

```python
# 1. 商品ごとの最大取引額
df.groupby("product_name")["amount"].max()

# 2. ぶどうの合計
df[df["product_name"] == "ぶどう (1パック)"]["amount"].sum()

# 3. 1月15日以降
df[df["date"] >= "2026-01-15"]

# 4. メロン or いちご、かつ数量5以上
df[
    df["product_name"].isin(["メロン", "いちご (1パック)"])
    & (df["quantity"] >= 5)
]
```

</details>

## よくあるエラー

### 1. 複数条件で `and` を使ってしまう
```python
df[(df["amount"] >= 1000) and (df["amount"] <= 5000)]  # ❌ エラー
```
→ pandas では `and` / `or` ではなく **`&` / `|`** を使います。
```python
df[(df["amount"] >= 1000) & (df["amount"] <= 5000)]  # ✅
```

### 2. 条件式を `()` で囲み忘れる
```python
df[df["amount"] >= 1000 & df["amount"] <= 5000]  # ❌ 演算子の優先順位で誤動作
```
→ **各条件を必ず `()` で囲む**。

### 3. `groupby` の結果を変数に保存して `print` したら見づらい
→ Jupyter (Colab) では、セルの **最後の行** だけ書けばキレイな表で表示されます。
```python
result = df.groupby("product_name")["amount"].sum()
result  # ← print(result) ではなくこう書く
```

### 4. `KeyError: 'amount'` が出る
→ 列名のタイプミス、または読み込んだ CSV/Excel に期待した列が無い。`df.columns` で実際の列名を確認しましょう。

## まとめ

- `df.groupby("列")["集計列"].sum()` で **ピボット集計**
- `.agg(["count", "sum", "mean"])` で **複数集計を同時**
- `df[ 条件 ]` で **行の絞り込み**（`&` `|` `isin` `between`）
- 1-4 の `for + if` 集計は、ほとんどが **pandas で 1〜2 行** になる

次は **2-4「データの加工」** で、新しい列を追加したり、欠損値や日付の処理を扱います。